## **Assignment 1: Generative Deep Learning - 708.088 (SS26)**
**Authored by:** Ozan Özdenizci, Institute of Machine Learning and Neural Computation, TU Graz

**Assignment issued on:** March 19th, 2026 09:00 AM\
**Assignment deadline:** April 16th, 2026 09:00 AM

# Training a VQ-VAE with an Autoregressive Prior

Vector-quantized variational autoencoders (VQ-VAE) combined with an autoregressive prior over the discrete latent space.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1).float())
])

train_data = torchvision.datasets.MNIST(root='data/', train=True, transform=transform, download=True)
test_data = torchvision.datasets.MNIST(root='data/', train=False, transform=transform, download=True)

train_data, val_data = torch.utils.data.random_split(train_data, [55000, 5000])

training_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

### Part (a): VQ-VAE architecture

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_latents):
        super(Encoder, self).__init__()
        self.num_latents = num_latents
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, num_latents * latent_dim)
        )

    def forward(self, x):
        h = self.net(x)
        return h.reshape(x.shape[0], self.num_latents, self.latent_dim)


class Decoder(nn.Module):
    def __init__(self, latent_dim, num_latents, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_latents * latent_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )

    def forward(self, z_q):
        B, L, D = z_q.shape
        z_q_flat = z_q.reshape(B, L * D)
        return self.net(z_q_flat)


class VectorQuantizer(nn.Module):
    def __init__(self, K, D, beta=0.25):
        super(VectorQuantizer, self).__init__()
        self.K = K          # number of codebook entries
        self.D = D          # dimensionality of each codebook entry
        self.beta = beta    # commitment loss weight

        # codebook with K entries each of dimension D
        self.embedding = nn.Embedding(K, D)
        self.embedding.weight.data.uniform_(-1.0 / K, 1.0 / K)

    def forward(self, z_e):
        B, L, D = z_e.shape   # batch size x latent sequence length x dim of codebook vectors

        # TODO: Compute distances between encoder outputs and codebook entries.
        # Find nearest codebook entry for each position
        # indices: codebook indices of shape (B, L)
        # z_q: quantized output, should be same shape as z_e
        # ...
        # ...
        indices = ...
        z_q = ...

        # TODO: Compute VQ loss: codebook loss + beta * commitment losss
        # Codebook loss: moves codebook entries toward encoder outputs
        # Commitment loss: moves encoder outputs toward codebook entries
        # ...
        vq_loss = ...

        # Straight-through estimator: copy gradients from z_q to z_e
        z_q = z_e + (z_q - z_e).detach()

        return z_q, indices.reshape(B, L), vq_loss

In [ ]:
class VQVAE(nn.Module):
    def __init__(self, encoder, decoder, quantizer):
        super(VQVAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.quantizer = quantizer

    def forward(self, x, reduction='avg'):
        z_e = self.encoder(x)
        z_q, indices, vq_loss = self.quantizer(z_e)
        x_recon = self.decoder(z_q)

        recon_loss = F.binary_cross_entropy(x_recon, x, reduction='none').sum(-1)

        if reduction == 'avg':
            return recon_loss.mean() + vq_loss
        elif reduction == 'sum':
            return recon_loss.sum() + vq_loss * x.shape[0]
        else:
            raise ValueError('reduction could be either `avg` or `sum`.')

    def encode_indices(self, x):
        with torch.no_grad():
            z_e = self.encoder(x)
            _, indices, _ = self.quantizer(z_e)
        return indices

    def decode_indices(self, indices):
        with torch.no_grad():
            z_q = self.quantizer.embedding(indices)
            x_recon = self.decoder(z_q)
        return x_recon

In [ ]:
def evaluation_vqvae(test_loader, model):
    model.eval()
    loss = 0.
    N = 0.
    for i, (test_batch, _) in enumerate(test_loader):
        loss_t = model.forward(test_batch, reduction='sum')
        loss = loss + loss_t.item()
        N = N + test_batch.shape[0]
    loss = loss / N
    return loss


def training_vqvae(name, num_epochs, model, optimizer, training_loader, val_loader):
    loss_val = []

    for ep in range(num_epochs):
        model.train()
        for i, (batch, _) in enumerate(training_loader):
            loss = model.forward(batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_loss = evaluation_vqvae(val_loader, model)
        loss_val.append(val_loss)

        print(f'Epoch: {ep}, val loss={val_loss:.4f}')
        torch.save(model, name + '.model')

    return np.asarray(loss_val)

### Part (b): VQ-VAE initialization, hyperparameters and training

In [ ]:
data_dim = 784      # input dimension (28x28 images flattened)
hidden_dim = 128    # hidden layer dimensionality
num_latents = 8     # number of discrete latent positions (latent sequence length L)
D = 16              # dimensionality of each codebook entry
K = 16              # codebook size (number of entries)
beta = 0.25         # commitment loss weight
lr = 1e-3
num_epochs_vqvae = 100

result_dir = 'results/'
if not(os.path.exists(result_dir)):
    os.mkdir(result_dir)
name = 'vqvae'

encoder = Encoder(input_dim=data_dim, hidden_dim=hidden_dim, latent_dim=D, num_latents=num_latents)
decoder = Decoder(latent_dim=D, num_latents=num_latents, hidden_dim=hidden_dim, output_dim=data_dim)
quantizer = VectorQuantizer(K=K, D=D, beta=beta)
vqvae = VQVAE(encoder, decoder, quantizer)

optimizer_vqvae = torch.optim.Adam([p for p in vqvae.parameters() if p.requires_grad == True], lr=lr)

nll_vqvae = training_vqvae(name=result_dir + name, num_epochs=num_epochs_vqvae, model=vqvae,
                           optimizer=optimizer_vqvae, training_loader=training_loader,
                           val_loader=val_loader)

In [ ]:
# TODO: Visualize the VQ-VAE model training process
# ...
# ...

# TODO: Visualize VQ-VAE model input versus reconstructions
# ...
# ...

# TODO: Visualize how many codebook entries are utilized across the training set.
# ...
# ...

## Part (c): Autoregressive prior over the latent space

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, A=False):
        super(CausalConv1d, self).__init__()
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.A = A
        self.padding = (kernel_size - 1) * dilation + A * 1

        self.conv1d = torch.nn.Conv1d(in_channels,
                                      out_channels,
                                      kernel_size,
                                      stride=1,
                                      padding=0,
                                      dilation=dilation,
                                      bias=True)

    def forward(self, x):
        x = torch.nn.functional.pad(x, (self.padding, 0))
        conv1d_out = self.conv1d(x)
        if self.A:
            return conv1d_out[:, :, :-1]
        else:
            return conv1d_out


class ARPrior(nn.Module):
    def __init__(self, net, num_latents, K):
        super(ARPrior, self).__init__()
        self.net = net      # causal convolutional network
        self.num_latents = num_latents    # length of the discrete latent sequence (L)
        self.K = K          # codebook size (number of possible values per position)
        self.EPS = 1.e-5

    def forward(self, indices, reduction='avg'):
        if reduction == 'avg':
            return -(self.log_prob(indices).mean())
        elif reduction == 'sum':
            return -(self.log_prob(indices).sum())
        else:
            raise ValueError('reduction could be either `avg` or `sum`.')

    def log_prob(self, indices):
        p = self.f(indices)
        indices_one_hot = F.one_hot(indices.long(), num_classes=self.K).float()
        log_p = indices_one_hot * torch.log(torch.clamp(p, self.EPS, 1. - self.EPS))
        log_p = torch.sum(log_p, dim=-1).sum(-1)
        return log_p

    def f(self, indices):
        x = indices.float().unsqueeze(1)
        h = self.net(x)
        h = h.permute(0, 2, 1)
        return torch.softmax(h, dim=-1)

    def sample(self, batch_size):
        indices = torch.zeros((batch_size, self.num_latents), dtype=torch.long)
        for l in range(self.num_latents):
            p = self.f(indices)
            indices[:, l] = torch.multinomial(p[:, l, :], num_samples=1).squeeze(1)
        return indices

In [ ]:
def evaluation_ar_prior(test_loader, vqvae, ar_prior):
    ar_prior.eval()
    loss = 0.
    N = 0.
    for i, (test_batch, _) in enumerate(test_loader):
        indices = vqvae.encode_indices(test_batch)
        loss_t = ar_prior.forward(indices, reduction='sum')
        loss = loss + loss_t.item()
        N = N + test_batch.shape[0]
    loss = loss / N
    return loss


def training_ar_prior(name, num_epochs, vqvae, ar_prior, optimizer, training_loader, val_loader):
    nll_val = []

    for ep in range(num_epochs):
        ar_prior.train()
        for i, (batch, _) in enumerate(training_loader):
            # TODO: Main training steps and loss computation
            # ...
            # ...

        val_loss = evaluation_ar_prior(val_loader, vqvae, ar_prior)
        nll_val.append(val_loss)
        print(f'Epoch: {ep}, val nll={val_loss:.4f}')

        # TODO: Generate and visualize 8 samples from the full model.
        # Sample index sequences from the AR prior, then decode with the frozen VQ-VAE decoder.
        # ...
        # ...

    return np.asarray(nll_val)

## Part (d): Autoregressive prior initialization, hyperparameters and training

In [ ]:
for p in vqvae.parameters():
    p.requires_grad = False
vqvae.eval()

arm_hdim = 128    # hidden layer dimensionality for AR prior
kernel = 3        # causal conv kernel size
lr_arm = 1e-3
num_epochs_arm = 150

prior_net = nn.Sequential(
    CausalConv1d(in_channels=1, out_channels=arm_hdim, dilation=1, kernel_size=kernel, A=True),
    nn.LeakyReLU(),
    CausalConv1d(in_channels=arm_hdim, out_channels=arm_hdim, dilation=1, kernel_size=kernel, A=False),
    nn.LeakyReLU(),
    CausalConv1d(in_channels=arm_hdim, out_channels=arm_hdim, dilation=1, kernel_size=kernel, A=False),
    nn.LeakyReLU(),
    CausalConv1d(in_channels=arm_hdim, out_channels=K, dilation=1, kernel_size=kernel, A=False)
)

ar_prior = ARPrior(prior_net, num_latents=num_latents, K=K)

optimizer_arm = torch.optim.Adam([p for p in ar_prior.parameters() if p.requires_grad == True], lr=lr_arm)

nll_prior = training_ar_prior(name=result_dir + name, num_epochs=num_epochs_arm, vqvae=vqvae,
                              ar_prior=ar_prior, optimizer=optimizer_arm,
                              training_loader=training_loader, val_loader=val_loader)

In [ ]:
# TODO: Visualize autoregressive prior model training process
# ...
# ...


In [ ]:
# TODO: Generate final samples from the full model.
# Sample index sequences from the AR prior, then decode with the frozen VQ-VAE decoder.
# ...
# ...

# TODO: Visualize the final samples from the full model.
# ...
# ...
